# Let's try some machine learning!

In [ ]:
import src.machine_learning as ML 
# Use original PNG size
dataset = ML.GlyphDataset('data/stars-large.zip')
dataset.show()

# Resize to 64x64 and show only 3 images
dataset_resized = ML.GlyphDataset('data/stars-and-letters.zip', resize=(64, 64))
dataset_resized.show(3)

## Create the DataLoader 

In [ ]:
import src.machine_learning as ML 
# 1. Create your dataset
dataset = ML.GlyphDataset('data/stars-large.zip')

# 2. Create DataLoader
loader1 = ML.create_loader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

#Creating loader 2 for same dataset
loader2 = ML.create_loader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)

# 3. Visualize the loaders
ML.visualize_loader(loader1)
ML.visualize_loader(loader2, max_images=10, nrow=5, silent=True)


## Create the Neural Network

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML

class GlyphClassifier(nn.Module):
    def __init__(self, NUM_classes):
        super(GlyphClassifier, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, NUM_classes)  

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # [B, 32, 32, 32]
        x = self.pool(F.relu(self.conv2(x)))  # [B, 64, 16, 16]
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x  


In [ ]:
NUM_classes = 10
# Train set
dataset = ML.GlyphDataset('data/simple-star.zip', resize=(64, 64), split = "train", num_classes=NUM_classes)
# Test set
test_dataset = ML.GlyphDataset('data/simple-star-test.zip', resize=(64, 64), split='test', num_classes=NUM_classes)  
# Test set 2
test_dataset_RS = ML.GlyphDataset('data/stars-small-test.zip', resize=(64,64), split = "test", num_classes=NUM_classes)

In [ ]:
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load dataset and dataloader
train_loader = ML.create_loader(dataset, batch_size=16, shuffle = True)

# Init model
model = GlyphClassifier(NUM_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 5
losses = []

for epoch in range(num_epochs):
    model.train()
    for images, labels, _ in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {losses[-1]:.4f}")


In [ ]:
ML.plot_training_loss(losses)

In [ ]:
test_loader = ML.create_loader(test_dataset, batch_size=16, shuffle = False)
model.eval()
correct_predictions = 0
total_samples = 0
with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        labels = labels.to(device) 

        outputs = model(images)

        _, predicted_labels = torch.max(outputs.data, 1)

        total_samples += labels.size(0) 
           
        correct_predictions += (predicted_labels == labels).sum().item()

accuracy = 100 * correct_predictions / total_samples
print(f'\n--- Evaluation Results ---')
print(f'Total Test Samples: {total_samples}')
print(f'Correct Predictions: {correct_predictions}')
print(f'Accuracy on the test set: {accuracy:.2f}%')
print(f'--------------------------')


In [ ]:

#  resnet-50, pre-trained on ImageNet
